# back-fn-call-with-recipe-args — worked example 2: sum_back fails without **recipe.kwargs (dim/keepdim)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `back-fn-call-with-recipe-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When the forward op took keyword arguments (like `t.sum(x, dim=1, keepdim=True)`), those kwargs are stored in `recipe.kwargs` and MUST be splatted with `**` into the back_fn. `sum_back` needs `dim` and `keepdim` to know how to broadcast `grad_out` back to the input's shape. Drop the `**` and the gradient comes out the wrong shape.

## Worked solution

**Goal.** Show that `sum_back` produces the correctly-shaped gradient only when the forward kwargs `dim`/`keepdim` reach it.

**Step 1 — the forward.** `out = x.sum(dim=1, keepdim=True)` turns a `(3, 4)` tensor into `(3, 1)`. The recipe records `args=(x,)` and `kwargs={'dim': 1, 'keepdim': True}`.

**Step 2 — what sum_back must do.** The derivative of a sum is broadcasting `grad_out` back over the reduced axis. With `keepdim=True`, `grad_out` is already `(3, 1)`, so a plain `broadcast_to(x.shape)` recovers `(3, 4)`. The back_fn needs `x` (for the target shape) — and in the general `keepdim=False` case it would need `dim` to re-insert the axis. So the signature is `sum_back(grad_out, out, x, dim, keepdim)`.

**Step 3 — the canonical call.** `sum_back(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. The `**` unpacks `dim=1, keepdim=True` into the matching keyword params. Without `**`, Python would raise a `TypeError` (missing `dim`) — the bug the registry pattern is designed to surface immediately.

**Why it works.** Every element of `x` contributed exactly once to its row sum, so each gets gradient `1 * grad_out_of_its_row`. Broadcasting `grad_out` of shape `(3, 1)` to `(3, 4)` is precisely that.

In [ ]:
def sum_back(grad_out, out, x, dim, keepdim):
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    return grad_out.broadcast_to(x.shape).clone()

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

t.manual_seed(0)
x = t.randn(3, 4)
out = x.sum(dim=1, keepdim=True)
node = Node(out, Recipe(t.sum, (x,), {'dim': 1, 'keepdim': True}))
grad_out = t.arange(3, dtype=t.float32).reshape(3, 1) + 1.0

dx = sum_back(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
print('dx shape:', tuple(dx.shape))
print('every row constant:', bool((dx == grad_out.broadcast_to(x.shape)).all()))